# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Not a dict, use dot notation

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")
print(f"Dataset Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and their field and column IDs.

In [ ]:
# List all RecordSet @ids (identify all major record containers in the dataset)

record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])] if hasattr(metadata, 'recordSet') else []
if not record_set_ids:
    print("No record sets listed directly on top-level metadata. Attempting to detect via hasPart or distribution...")
    # fallback: Some datasets include the main RecordSet as the distribution; let's check
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            if '@id' in dist:
                print(f"  Distribution recordset candidate: {dist['@id']}")
    else:
        print("No record sets or distributions found in metadata.")
else:
    for rsid in record_set_ids:
        print(f"  RecordSet: {rsid}")

print("\nAttempting to retrieve field and column info for each record set")

# To make this reliable, enumerate the record sets from dataset._croissant['recordSet'], if populated.
croissant_dict = dataset._croissant

if 'recordSet' in croissant_dict and len(croissant_dict['recordSet']) > 0:
    for rs in croissant_dict['recordSet']:
        rsid = rs.get('@id', 'N/A')
        print(f"\nRecordSet: {rsid}")
        # List associated columns:
        if 'column' in rs and rs['column']:
            for col in rs['column']:
                col_id = col.get('@id', 'N/A')
                print(f"  Column: {col_id} - Name: {col.get('name', '')}")
        # List associated fields:
        if 'field' in rs and rs['field']:
            for field in rs['field']:
                field_id = field.get('@id', 'N/A')
                print(f"  Field: {field_id} - Name: {field.get('name', '')}")
else:
    # Fallback: If no record sets in croissant, list fields from top-level as available
    if hasattr(metadata, 'field'):
        for field in getattr(metadata, 'field'):
            print(f"  Field: {field['@id']} - Name: {field.get('name', '')}")
    else:
        print("No record sets or fields detected in the schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find main record sets.
main_record_sets = []
if 'recordSet' in croissant_dict and len(croissant_dict['recordSet']) > 0:
    for rs in croissant_dict['recordSet']:
        rsid = rs.get('@id', '')
        main_record_sets.append(rsid)

# If none specified, try guessing from 'distribution'
if not main_record_sets:
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            main_record_sets.append(dist['@id'])

# If still not found, set empty
if not main_record_sets:
    print("No record sets available for extraction.")
else:
    print("Main record sets found:")
    for rs in main_record_sets:
        print(f"  {rs}")

# Extract dataframes for each record set
dataframes = {}
for rsid in main_record_sets:
    try:
        records = list(dataset.records(record_set=rsid))
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet {rsid}")
    except Exception as e:
        print(f"Could not load records for {rsid}: {str(e)}")

# Preview columns for the first available record set
chosen_rs = main_record_sets[0] if len(main_record_sets) > 0 else None
if chosen_rs and chosen_rs in dataframes and not dataframes[chosen_rs].empty:
    print(f"RecordSet: {chosen_rs} columns:")
    print(dataframes[chosen_rs].columns.tolist())
    display(dataframes[chosen_rs].head())
else:
    print("No data loaded to preview.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration purposes, we select a typical numeric field from proteomics (e.g. 'cr:MW' or 'cr:coverage' or 'cr:peptide_count') as numeric_field_id.

import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

if chosen_rs and chosen_rs in dataframes and not dataframes[chosen_rs].empty:
    df = dataframes[chosen_rs]
    # Try to guess likely numeric field: MW, coverage, or normalized abundance
    possible_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ['mw','weight','coverage','abundance','count','number','intensity','score','percent','peptide'])]
    if not possible_numeric_fields:
        possible_numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
    print(f"Using numeric field: {numeric_field}")

    # Show distribution and basic stats
    summary = df[numeric_field].describe()
    print(f"\nSummary statistics for {numeric_field}:")
    print(summary)

    # Remove extreme outliers (e.g., above 99th percentile)
    thresh = df[numeric_field].quantile(0.99)
    filtered_df = df[df[numeric_field] < thresh]
    print(f"\nFiltered outliers above {thresh:.2f} in {numeric_field}. Remaining records: {len(filtered_df)}")

    # Normalize (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nSample of normalized data:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely categorical field ('cr:accession', 'cr:description', or the first string column)
    possible_group_fields = [col for col in df.columns if any(s in col.lower() for s in ['group','type','class','accession','description','category','id','sample']) and col != numeric_field]
    if not possible_group_fields:
        possible_group_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = possible_group_fields[0] if possible_group_fields else None

    if group_field:
        grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped mean statistics by {group_field} (showing up to 5 groups):")
        print(grouped[[numeric_field]].head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs and chosen_rs in dataframes and not dataframes[chosen_rs].empty:
    df = dataframes[chosen_rs]
    # Use same field as above
    num_field = numeric_field if 'numeric_field' in locals() else df.select_dtypes(include=np.number).columns.tolist()[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[num_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {num_field}")
    plt.xlabel(num_field)
    plt.ylabel("Frequency")
    plt.show()

    # If possible, show boxplot grouped by the group field
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        # Plot only top 5 categories by count
        top_groups = df[group_field].value_counts().nlargest(5).index
        sns.boxplot(x=group_field, y=num_field, data=df[df[group_field].isin(top_groups)])
        plt.title(f"{num_field} by {group_field} (top 5 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR² Croissant dataset using `mlcroissant` and `pandas`. You loaded the dataset metadata, explored available record sets and their fields by unique `@id`, extracted data into DataFrames, filtered and normalized data, and performed simple exploratory visualizations. This dataset enables detailed analysis of protein abundance and related attributes in human extracellular vesicles using standardized, FAIR-aligned workflows.

Further analysis can leverage the clean, structured tabular data for biomarker discovery, protein characterization, or data integration with other proteomics studies.